# 04 — create_task: Running Things in the Background

**Goal:** Understand `asyncio.create_task()`, `asyncio.Event`, and how multiple long-running loops coexist.

So far you know:
- `await` pauses your function, lets others run
- `asyncio.gather()` runs multiple coroutines concurrently

But what if you want to start something in the background and NOT wait for it? That's `create_task`.

## Exercise 4.1 — await vs create_task

In [ ]:
import asyncio
import time

async def slow_job(name, seconds):
    print(f"[{time.time():.1f}] {name}: started")
    await asyncio.sleep(seconds)
    print(f"[{time.time():.1f}] {name}: done")

# Version 1: await each one (SEQUENTIAL)
print("--- Sequential ---")
start = time.time()
await slow_job("A", 1)  # wait for A to finish
await slow_job("B", 1)  # THEN start B
print(f"Total: {time.time() - start:.1f}s\n")

# Version 2: create_task (CONCURRENT)
print("--- Concurrent ---")
start = time.time()
task_a = asyncio.create_task(slow_job("A", 1))  # start A in background
task_b = asyncio.create_task(slow_job("B", 1))  # start B in background
await task_a  # now wait for both
await task_b
print(f"Total: {time.time() - start:.1f}s")

### Question 4.1
Sequential = 2s, concurrent = 1s. `create_task` starts a coroutine running WITHOUT waiting for it. When does it actually start running?

*Your answer:*



## Exercise 4.2 — Background loops (like sbot's agent_loop)

In sbot, `agent_loop` runs forever in the background, processing messages. Here's a simplified version:

In [ ]:
queue = asyncio.Queue()

async def background_worker():
    """Runs forever, processing items from queue."""
    while True:
        item = await queue.get()  # waits here when queue is empty
        print(f"  Worker: processing '{item}'...")
        await asyncio.sleep(0.5)  # simulate work
        print(f"  Worker: done with '{item}'")

# Start worker in background
worker_task = asyncio.create_task(background_worker())

# Feed it items
for msg in ["hello", "world", "bye"]:
    await queue.put(msg)
    print(f"Main: sent '{msg}'")
    await asyncio.sleep(1)  # wait between sends

await asyncio.sleep(1)  # let worker finish last item
worker_task.cancel()  # stop the background loop
print("Main: worker cancelled")

### Question 4.2
The worker runs `while True` — it never returns. How does the program not hang? What does `task.cancel()` do?

*Your answer:*



## Exercise 4.3 — asyncio.Event: signaling between tasks

Sometimes you need one task to WAIT until another task signals "I'm done."

`asyncio.Event` is like a flag:
- `event.wait()` → blocks until the flag is raised
- `event.set()` → raises the flag (unblocks all waiters)
- `event.clear()` → lowers the flag (for reuse)

In [ ]:
done = asyncio.Event()

async def cook():
    print("Cook: preparing food...")
    await asyncio.sleep(2)
    print("Cook: food is ready!")
    done.set()  # raise the flag

async def customer():
    print("Customer: waiting for food...")
    await done.wait()  # blocked here until flag is raised
    print("Customer: yum!")

done.clear()
await asyncio.gather(cook(), customer())

## Exercise 4.4 — YOUR TURN: mini chat system

Build a mini version of sbot's architecture:
- A background `agent` task that reads from a queue and "responds"
- A `cli` function that sends messages and waits for each response using `asyncio.Event`

In [ ]:
inbound = asyncio.Queue()
response_event = asyncio.Event()
last_response = ""

async def agent():
    global last_response
    while True:
        msg = await inbound.get()
        print(f"  Agent: thinking about '{msg}'...")
        await asyncio.sleep(1)
        last_response = f"Reply to: {msg}"
        # TODO: signal that response is ready
        # ???

async def cli():
    for msg in ["hello", "what is async", "bye"]:
        response_event.clear()
        await inbound.put(msg)
        print(f"CLI: sent '{msg}'")
        # TODO: wait for agent to respond
        # ???
        print(f"CLI: got '{last_response}'\n")

agent_task = asyncio.create_task(agent())
await cli()
agent_task.cancel()

## Summary

| What | How | When to use |
|------|-----|-------------|
| `await coro()` | Run and wait for result | You need the result now |
| `create_task(coro())` | Start in background | Long-running loops, fire-and-forget |
| `task.cancel()` | Stop a background task | Shutdown |
| `asyncio.Event` | Signal between tasks | "Notify me when you're done" |

**Next notebook:** Reproduce and fix the exact sbot buffering bug